# Módulo 15 · Práctica: construye tu GPT

Entrenas en tu CPU un **mini-GPT de caracteres** de 107.694 parámetros con la receta de los LLM: ventanas de texto, Transformer decoder causal, AdamW con warmup y cosine, y muestreo. Grok-1, con pesos públicos, tiene unos 3 millones de veces más parámetros.

Completa las celdas `# TU CÓDIGO` y ejecuta la `# Comprobación` de detrás. Todo tarda 1–3 minutos.

| Parte | Contenido | Parte | Contenido |
| --- | --- | --- | --- |
| 0 | Entorno | 5 | Warmup y cosine (ej. 4) |
| 1 | Corpus y tokenizador (ej. 1) | 6 | Entrenamiento y puntos de control |
| 2 | Ventanas y batches (ej. 2) | 7 | Curvas de pérdida |
| 3 | Bigrama de referencia | 8 | Temperatura y top-k (ej. 5) |
| 4 | Mini-GPT (ej. 3: atención) | 9 | Mapas de atención y retos |

## Parte 0 · Entorno

Semillas fijas para reproducir los números del documento y un solo hilo de CPU (con modelos tan pequeños, más hilos no aceleran).

In [ ]:
import math, time, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.set_num_threads(1)
torch.manual_seed(1337)
print("PyTorch", torch.__version__)

## Parte 1 · El corpus y el tokenizador

No descargamos nada: generamos 200 líneas originales con plantillas y azar (partes del tiempo, recetas, diálogos y objetos inventados). Como conoces la estructura exacta, puedes juzgar qué aprende el modelo.

In [ ]:
def generar_corpus(semilla=15, n_frases=200):
    r = random.Random(semilla)
    L = lambda s: s.split(",")   # "a,b,c" -> ["a", "b", "c"]
    ciudades = L("villamar,puerto alto,sierra fría,valdelobos,costa clara,monteverde,río seco,la llanura")
    cielos = L("despejado,nublado,con niebla,con lluvia débil,con tormenta,cubierto")
    vientos = L("flojo del norte,moderado del sur,fuerte del oeste,en calma")
    ingredientes = L("arroz,lentejas,patata,calabaza,tomate,cebolla,pimiento,garbanzos,berenjena,zanahoria,queso,pan")
    verbos = L("cuece,sofríe,asa,mezcla,hornea,tritura")
    minutos = L("cinco,diez,quince,veinte,treinta")
    nombres = L("ana,luis,marta,pablo,irene,jorge,nuria,tomás")
    objetos = L("una taza,una lámpara,una mochila,una silla,un reloj,una caja")
    colores = L("roja,azul,verde,blanca,negra,amarilla")
    materiales = L("madera,metal,cristal,tela,barro,papel")
    usos = L("guardar semillas,leer de noche,llevar libros,esperar sentado,medir el tiempo,ordenar cables")
    lugares = L("la cocina,el mercado,la estación,el parque,la biblioteca")
    respuestas = ["sí, ahora mismo", "no, mañana", "mejor después de comer", "vale, te espero"]
    masc = {"roja": "rojo", "blanca": "blanco", "negra": "negro", "amarilla": "amarillo"}

    def tiempo():
        c = r.choice(ciudades)
        mn, mx = sorted(r.sample(range(2, 31), 2))
        return (f"parte del tiempo. en {c} el cielo estará {r.choice(cielos)}. "
                f"mínima de {mn} grados y máxima de {mx}. viento {r.choice(vientos)}.")

    def receta():
        a, b = r.sample(ingredientes, 2)
        return (f"receta. {r.choice(verbos)} {a} con {b} durante {r.choice(minutos)} minutos. "
                f"añade sal y sirve {r.choice(['caliente', 'templado', 'frío'])}.")

    def dialogo():
        p, q = r.sample(nombres, 2)
        return f"diálogo. {p}: ¿vamos a {r.choice(lugares)}? {q}: {r.choice(respuestas)}."

    def objeto():
        o, col = r.choice(objetos), r.choice(colores)
        if o.startswith("un "):          # concordancia de género: un reloj negro
            col = masc.get(col, col)
        return f"objeto. {o} {col} de {r.choice(materiales)}. sirve para {r.choice(usos)}."

    tipos = [tiempo, receta, dialogo, objeto]
    return "\n".join(r.choice(tipos)() for _ in range(n_frases)) + "\n"


texto = generar_corpus()
print(len(texto), "caracteres")
print(texto[:400])

**Ejercicio 1.** Construye el vocabulario de caracteres (ordenado), los diccionarios `stoi` y `itos` y las funciones `codificar(s)` → lista de enteros y `decodificar(lista)` → cadena.

In [ ]:
# TU CÓDIGO
chars = None          # lista ordenada de caracteres distintos de texto
V = None              # tamaño del vocabulario
stoi, itos = {}, {}

def codificar(s):
    return None

def decodificar(ids):
    return None

In [ ]:
# Comprobación
assert V == 46, f"Deberías tener 46 caracteres distintos, tienes {V}"
assert decodificar(codificar("receta. asa pan")) == "receta. asa pan", "codificar y decodificar deben ser inversas"
assert codificar("\n")[0] == 0, "Con el orden de sorted, el salto de línea es el token 0"
print("Correcto. Vocabulario:", repr("".join(chars)))

## Parte 2 · Ventanas de contexto y batches

El 90 % inicial entrena y el 10 % final valida. Un ejemplo es una **ventana** `x` de `T` caracteres y la misma desplazada una posición, `y[t] = x[t+1]`: **T predicciones** a la vez.

In [ ]:
data = torch.tensor(codificar(texto), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(len(train_data), "tokens de entrenamiento ·", len(val_data), "de validación")

T = 32   # longitud de contexto (block size)
B = 32   # ventanas por batch

**Ejercicio 2.** Escribe `get_batch(split, g=None)`: elige `B` posiciones iniciales al azar con `torch.randint(len(d) - T, (B,), generator=g)` y devuelve `x` e `y`, ambos de forma `(B, T)`, con `y` desplazado una posición.

In [ ]:
def get_batch(split, g=None):
    d = train_data if split == "train" else val_data
    # TU CÓDIGO
    x, y = None, None
    return x, y

In [ ]:
# Comprobación
xb, yb = get_batch("train")
assert xb.shape == (B, T) and yb.shape == (B, T), "x e y deben tener forma (B, T)"
assert torch.equal(xb[:, 1:], yb[:, :-1]), "y debe ser x desplazado una posición"
print("Correcto. Primera ventana:")
print(repr(decodificar(xb[0].tolist())))
print(repr(decodificar(yb[0].tolist())))

## Parte 3 · La línea base: un bigrama

El listón: el bigrama del módulo 1 (suavizado α = 0,1). Si el GPT no lo supera con claridad, algo va mal.

In [ ]:
N = torch.zeros(V, V)
for a, b in zip(train_data[:-1], train_data[1:]):
    N[a, b] += 1
P = (N + 0.1) / (N + 0.1).sum(1, keepdim=True)
bigrama_val = -torch.log(P[val_data[:-1], val_data[1:]]).mean().item()
print(f"Uniforme:          {math.log(V):.3f} nats/carácter")
print(f"Bigrama (val):     {bigrama_val:.3f} nats/carácter")

## Parte 4 · La arquitectura mini-GPT

El modelo es exactamente el bloque del módulo 14, en versión **pre-norm**:

$$x \leftarrow x + \mathrm{MHA}(\mathrm{LN}(x)), \qquad x \leftarrow x + \mathrm{MLP}(\mathrm{LN}(x))$$

y la atención causal del módulo 13, con H cabezas de dimensión $d_k = d/H$:

$$\mathrm{Atención}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V, \qquad M_{ts} = \begin{cases}0 & s \le t\\ -\infty & s > t\end{cases}$$

**Ejercicio 3.** Completa `forward` de `AtencionCausal`: calcula las puntuaciones, aplica la máscara con `masked_fill` y la softmax, y guarda los pesos en `self.ultimos_pesos` (los dibujaremos en la parte 9).

In [ ]:
class AtencionCausal(nn.Module):
    def __init__(self, d, h, T):
        super().__init__()
        self.h = h
        self.qkv = nn.Linear(d, 3 * d, bias=False)   # Q, K, V de todas las cabezas
        self.proj = nn.Linear(d, d)                  # W_O
        self.register_buffer("mask", torch.tril(torch.ones(T, T)).bool())
        self.ultimos_pesos = None

    def forward(self, x):
        Bb, Tt, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, h, T, dk): cada cabeza trabaja por separado
        q, k, v = [t.view(Bb, Tt, self.h, C // self.h).transpose(1, 2) for t in (q, k, v)]
        # TU CÓDIGO: puntuaciones (B, h, T, T), máscara causal, softmax
        pesos = None
        self.ultimos_pesos = pesos
        out = pesos @ v                                   # (B, h, T, dk)
        out = out.transpose(1, 2).reshape(Bb, Tt, C)      # concatenar cabezas
        return self.proj(out)

In [ ]:
# Comprobación: cambiar el futuro no puede cambiar el pasado
torch.manual_seed(0)
att = AtencionCausal(16, 4, 8)
x1 = torch.randn(1, 8, 16); x2 = x1.clone(); x2[:, 5:] = torch.randn(1, 3, 16)
o1, o2 = att(x1), att(x2)
assert torch.allclose(o1[:, :5], o2[:, :5], atol=1e-6), "Las posiciones 0-4 no deben ver las 5-7: revisa la máscara"
assert torch.allclose(att.ultimos_pesos.sum(-1), torch.ones(1, 4, 8)), "Cada fila de pesos debe sumar 1"
assert att.ultimos_pesos[0, 0, 0, 1:].abs().max() == 0, "La posición 0 solo puede atenderse a sí misma"
print("Correcto: la atención es causal")

El resto ya está escrito: embeddings de token y de posición (aprendidos, como en GPT-2), L bloques, LayerNorm final y cabeza lineal.

In [ ]:
class Bloque(nn.Module):
    def __init__(self, d, h, T):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.att = AtencionCausal(d, h, T)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, V, d=64, h=4, L=2, T=32):
        super().__init__()
        self.T = T
        self.tok = nn.Embedding(V, d)
        self.pos = nn.Embedding(T, d)
        self.bloques = nn.Sequential(*[Bloque(d, h, T) for _ in range(L)])
        self.lnf = nn.LayerNorm(d)
        self.cabeza = nn.Linear(d, V)

    def forward(self, idx, y=None):
        Tt = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(Tt))
        logits = self.cabeza(self.lnf(self.bloques(x)))
        if y is None:
            return logits, None
        return logits, F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

torch.manual_seed(1337)
modelo = MiniGPT(V, d=64, h=4, L=2, T=T)
n_param = sum(p.numel() for p in modelo.parameters())
print(f"Parámetros: {n_param:,}".replace(",", "."))
_, l0 = modelo(*get_batch("train"))
print(f"Pérdida inicial: {l0.item():.3f}   (ln V = {math.log(V):.3f})")

In [ ]:
# Comprobación
d, L = 64, 2
esperado = V*d + T*d + L*(12*d*d + 10*d) + 2*d + d*V + V
assert n_param == esperado == 107694, f"Recuento esperado 107.694, sale {n_param}"
assert abs(l0.item() - math.log(V)) < 0.5, "Al empezar, la pérdida debe estar cerca de ln V"
print("Correcto: el recuento coincide con la fórmula del documento")

## Parte 5 · Tasa de aprendizaje: warmup y cosine

Planificador del módulo 7, con $\eta_{\min} = \eta_{\max}/10$:

$$\eta(t) = \begin{cases} \eta_{\max}\,\dfrac{t+1}{W} & t < W\\[2mm] \eta_{\min} + \tfrac12(\eta_{\max}-\eta_{\min})\left(1+\cos\dfrac{\pi (t-W)}{S-W}\right) & t \ge W\end{cases}$$

**Ejercicio 4.** Implementa `lr_en(t)`.

In [ ]:
PASOS, WARMUP, LR_MAX = 800, 100, 3e-3
LR_MIN = LR_MAX / 10

def lr_en(t):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
assert abs(lr_en(0) - 3e-5) < 1e-12, "En t = 0 la tasa es lr_max/warmup"
assert abs(lr_en(99) - 3e-3) < 1e-12 and abs(lr_en(100) - 3e-3) < 1e-12, "El pico está al final del warmup"
assert abs(lr_en(450) - 1.65e-3) < 1e-9, "A mitad del coseno: (lr_max + lr_min)/2"
plt.figure(figsize=(6, 2.5)); plt.plot([lr_en(t) for t in range(PASOS)])
plt.xlabel("paso"); plt.ylabel("tasa"); plt.title("Warmup + cosine"); plt.show()
print("Correcto")

## Parte 6 · Bucle de entrenamiento

Cada paso: tasa, batch, pérdida, `zero_grad`, `backward`, recorte de gradiente y `step`. Cada 100 pasos evaluamos con 20 batches fijos y guardamos un **punto de control** si la validación mejora.

In [ ]:
@torch.no_grad()
def estimar_perdida(modelo, n_batches=20):
    modelo.eval()
    g = torch.Generator().manual_seed(0)      # batches fijos
    out = {}
    for split in ["train", "val"]:
        out[split] = sum(modelo(*get_batch(split, g))[1].item() for _ in range(n_batches)) / n_batches
    modelo.train()
    return out

torch.manual_seed(1337)
modelo = MiniGPT(V, d=64, h=4, L=2, T=T)
opt = torch.optim.AdamW(modelo.parameters(), lr=LR_MAX, weight_decay=0.1)
historial, mejor_val, t0 = [], float("inf"), time.time()

for paso in range(PASOS):
    for grupo in opt.param_groups:
        grupo["lr"] = lr_en(paso)
    xb, yb = get_batch("train")
    _, perdida = modelo(xb, yb)
    opt.zero_grad(set_to_none=True)
    perdida.backward()
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
    opt.step()
    if (paso + 1) % 100 == 0:
        e = estimar_perdida(modelo)
        historial.append((paso + 1, e["train"], e["val"]))
        if e["val"] < mejor_val:
            mejor_val = e["val"]
            torch.save({"modelo": modelo.state_dict(), "opt": opt.state_dict(),
                        "paso": paso + 1, "val": mejor_val}, "mini_gpt.pt")
        print(f"paso {paso+1:4d} · train {e['train']:.3f} · val {e['val']:.3f} · {time.time()-t0:5.1f} s")

In [ ]:
# Comprobación
assert mejor_val < 0.6 * bigrama_val, f"La validación ({mejor_val:.3f}) debería quedar muy por debajo del bigrama ({bigrama_val:.3f})"
ck = torch.load("mini_gpt.pt")
modelo2 = MiniGPT(V, d=64, h=4, L=2, T=T); modelo2.load_state_dict(ck["modelo"])
assert abs(estimar_perdida(modelo2)["val"] - ck["val"]) < 1e-5, "El punto de control debe reproducir su pérdida"
print(f"Correcto. Mejor val {mejor_val:.3f} en el paso {ck['paso']}; bigrama {bigrama_val:.3f}")

## Parte 7 · Curvas de pérdida

Pregúntate: ¿dónde alcanza al bigrama?, ¿cuándo empieza a usar el contexto largo?, ¿cuánto se separan train y val (sobreajuste)?

In [ ]:
pasos_h, tr_h, va_h = zip(*historial)
plt.figure(figsize=(7, 4))
plt.plot(pasos_h, tr_h, "o-", label="entrenamiento")
plt.plot(pasos_h, va_h, "s-", label="validación")
plt.axhline(bigrama_val, ls="--", c="gray", label="bigrama")
plt.axhline(math.log(V), ls=":", c="gray", label="uniforme")
plt.xlabel("paso"); plt.ylabel("pérdida (nats/carácter)"); plt.legend(); plt.show()
print(f"Perplejidad final de validación: {math.exp(va_h[-1]):.2f}  (bigrama: {math.exp(bigrama_val):.2f})")

## Parte 8 · Generación con temperatura y top-k

Repetir: contexto de los últimos `T` tokens, logits de la última posición, dividir por la temperatura, dejar los `k` mayores (resto a −∞), softmax y muestrear.

**Ejercicio 5.** Completa `generar`.

In [ ]:
@torch.no_grad()
def generar(modelo, inicio, n_nuevos=200, temperatura=1.0, top_k=None, semilla=0):
    g = torch.Generator().manual_seed(semilla)
    modelo.eval()
    idx = torch.tensor([codificar(inicio)])
    for _ in range(n_nuevos):
        contexto = idx[:, -modelo.T:]
        logits, _ = modelo(contexto)
        logits = logits[:, -1, :]                  # (1, V)
        # TU CÓDIGO: temperatura, top-k y muestreo con torch.multinomial(probs, 1, generator=g)
        siguiente = None
        idx = torch.cat([idx, siguiente], dim=1)
    modelo.train()
    return decodificar(idx[0].tolist())

In [ ]:
# Comprobación
modelo.load_state_dict(torch.load("mini_gpt.pt")["modelo"])   # usamos el mejor punto de control
m1 = generar(modelo, "receta.", 120, temperatura=1.0, top_k=1)
m2 = generar(modelo, "receta.", 120, temperatura=1.0, top_k=1, semilla=99)
assert m1 == m2, "Con top_k = 1 la generación es voraz: no depende de la semilla"
s = generar(modelo, "\n", 300, temperatura=0.8, top_k=10)
assert len(s) == 301 and set(s) <= set(chars), "Debe generar 300 caracteres del vocabulario"
print("Correcto. Muestra (T = 0,8, top-k = 10):\n")
print(s)

In [ ]:
for temp in [0.5, 1.0, 1.5, 3.0]:
    print(f"--- temperatura {temp} ---")
    print(generar(modelo, "objeto.", 160, temperatura=temp, semilla=1), "\n")

**Pregunta.** ¿Por qué con temperatura 3 aparecen palabras inexistentes (Boltzmann, módulo 1)? ¿Respeta el modelo la concordancia «un reloj negro» / «una taza negra»?

## Parte 9 · Mapas de atención y retos

Pesos de las 4 cabezas del último bloque. Busca cabezas que miren al carácter anterior (diagonal) o a la etiqueta inicial («objeto»), que decide la plantilla.

In [ ]:
frase = "objeto. un reloj verde de metal."
with torch.no_grad():
    modelo.eval(); modelo(torch.tensor([codificar(frase)])); modelo.train()
pesos = modelo.bloques[-1].att.ultimos_pesos[0]          # (h, T, T)
fig, axs = plt.subplots(1, 4, figsize=(16, 4.5))
for c in range(4):
    axs[c].imshow(pesos[c], cmap="Blues")
    axs[c].set_title(f"cabeza {c}")
    axs[c].set_xticks(range(len(frase))); axs[c].set_xticklabels(list(frase), fontsize=7)
    axs[c].set_yticks(range(len(frase))); axs[c].set_yticklabels(list(frase), fontsize=7)
plt.tight_layout(); plt.show()

**Retos** (compara con la tabla de la sección 15.7):

1. **Tasa de aprendizaje.** Repite con `LR_MAX` = `3e-4`, `3e-2` y `3e-1`. ¿Cuál se queda corta, cuál mejora y cuál es inestable?
2. **Contexto.** Pon `T = 8`. El parte del tiempo dice «mínima de 4 grados y máxima de 23»: ¿puede el modelo saber que está en un parte del tiempo si no ve la etiqueta?
3. **Tamaño.** Prueba `L = 1` y `d = 128, L = 3`. ¿Compensa el coste extra frente a subir la tasa?
4. **Límite teórico.** El corpus tiene una entropía mínima de unos 0,125 nats/carácter (sección 15.7). ¿Cuánto le falta a tu modelo?
5. **Hito 3.** Informe con curva de pérdida, tres muestras y un mapa de atención interpretado.

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 1
# chars = sorted(set(texto)); V = len(chars)
# stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for i, c in enumerate(chars)}
# codificar = lambda s: [stoi[c] for c in s];  decodificar = lambda ids: "".join(itos[i] for i in ids)
# Ejercicio 2
# ix = torch.randint(len(d) - T, (B,), generator=g)
# x = torch.stack([d[i:i + T] for i in ix]); y = torch.stack([d[i + 1:i + T + 1] for i in ix])
# Ejercicio 3
# punt = q @ k.transpose(-2, -1) / math.sqrt(C // self.h)
# punt = punt.masked_fill(~self.mask[:Tt, :Tt], float("-inf"))
# pesos = punt.softmax(dim=-1)
# Ejercicio 4
# if t < WARMUP: return LR_MAX * (t + 1) / WARMUP
# prog = (t - WARMUP) / (PASOS - WARMUP)
# return LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1 + math.cos(math.pi * prog))
# Ejercicio 5
# logits = logits[:, -1, :] / temperatura
# if top_k is not None:
#     umbral = torch.topk(logits, top_k).values[:, -1:]
#     logits = logits.masked_fill(logits < umbral, float("-inf"))
# siguiente = torch.multinomial(logits.softmax(dim=-1), 1, generator=g)